In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
recipe = pd.read_json("train.json")
recipe.head()

,id,cuisine,ingredients
0,10259,greek,"[romaine lettuce, black olives, grape tomatoes..."
1,25693,southern_us,"[plain flour, ground pepper, salt, tomatoes, g..."
2,20130,filipino,"[eggs, pepper, salt, mayonaise, cooking oil, g..."
3,22213,indian,"[water, vegetable oil, wheat, salt]"
4,13162,indian,"[black pepper, shallots, cornflour, cayenne pe..."


In [3]:
import json
with open("train.json") as f:
    data = json.load(f)

In [4]:
print(recipe.shape)
print(recipe.columns)
print(recipe["cuisine"].nunique(), "cuisines")

(39774, 3)
Index(['id', 'cuisine', 'ingredients'], dtype='object')
20 cuisines


In [5]:
#Preprocessing
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)

In [6]:
recipe["ingredients_text"] = recipe["ingredients"].apply(lambda x: " ".join(x))
X = recipe["ingredients_text"]
y = recipe["cuisine"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [9]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5, metric="cosine")
model.fit(X_train_tfidf, y_train)

KNeighborsClassifier(metric='cosine')

In [10]:
start = time.time()
model.fit(X_train_tfidf, y_train)
train_time = time.time() - start

start = time.time()
pred = model.predict(X_test_tfidf)
pred_time = time.time() - start

print("Training time (s):", round(train_time, 3))
print("Prediction time (s):", round(pred_time, 3))

Training time (s): 0.385
Prediction time (s): 16.707


In [11]:
y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.6927718416090509


In [13]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

   brazilian       0.51      0.53      0.52        93
     british       0.35      0.41      0.38       161
cajun_creole       0.63      0.65      0.64       309
     chinese       0.66      0.84      0.74       535
    filipino       0.57      0.48      0.53       151
      french       0.48      0.54      0.51       529
       greek       0.61      0.52      0.56       235
      indian       0.79      0.81      0.80       601
       irish       0.58      0.35      0.43       133
     italian       0.69      0.83      0.76      1568
    jamaican       0.79      0.58      0.67       105
    japanese       0.71      0.60      0.65       284
      korean       0.74      0.67      0.70       166
     mexican       0.84      0.85      0.84      1288
    moroccan       0.74      0.54      0.62       164
     russian       0.71      0.20      0.32        98
 southern_us       0.71      0.66      0.68       864
     spanish       0.66    

In [14]:
def predict_cuisine(ingredients):

    text = " ".join(ingredients).lower()

    vec = tfidf.transform([text])

    prediction = model.predict(vec)

    return prediction[0]

In [15]:
predict_cuisine(["spinach","cheese","olives"])

'italian'